# YOLOv11 推理 Notebook
下面演示如何加载单张图片并逐步推理

## 1. 环境准备 & 导入包
导入必要的 Python 包

In [ ]:
from ultralytics import YOLO            # YOLOv11 模型接口
import matplotlib.pyplot as plt         # 用于图像展示
import cv2                              # 用于图像读取与绘制
import os                               # 用于文件和路径操作
import matplotlib                       # 用于配置字体

# 配置 matplotlib 支持中文，避免中文标签或标题出现缺失
matplotlib.rcParams['font.sans-serif'] = ['SimHei']  # 设置中文字体为黑体
matplotlib.rcParams['axes.unicode_minus'] = False    # 解决负号 '-' 显示为方块的问题

print("环境准备完成：依赖包已导入，已配置中文字体支持")


## 2. 加载模型
从训练输出目录加载最佳权重

In [ ]:
model = YOLO('runs/train/exp/weights/best.pt')  # 加载训练好的 .pt 文件
print(f"模型加载完成。\n类别名称：{model.names}")

## 3. 读取并展示输入图片
- 指定单张图片路径，确认文件存在
- 用 OpenCV 读取 BGR 图像并转换为 RGB
- 使用 matplotlib 展示

In [ ]:
img_path = 'datasets/weeds/images/train/02981.jpg'  # 修改为你的图片路径
assert os.path.exists(img_path), f"错误：未找到图片：{img_path}"

# 读取图像（BGR 格式）
img_bgr = cv2.imread(img_path)
print(f"加载图像，原始形状 (高, 宽, 通道)：{img_bgr.shape}")

# 转换为 RGB 以便 matplotlib 正确显示
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(6,6))
plt.axis('off')
plt.title('输入图像')
plt.imshow(img_rgb)
plt.show()

## 4. 对单张图片执行推理
- 使用 `model.predict` 获取 `Results` 对象列表
- 设置 `save=False` 和 `verbose=False` 关闭文件保存与进度输出

In [ ]:
results = model.predict(
    source=img_path,    # 输入单张图片路径
    imgsz=640,          # 调整推理尺寸
    save=False,         # 不将结果保存为新文件
    verbose=False       # 不输出额外日志
)
print(f"推理完成。结果数量：{len(results)}")

# 获取第一个结果对象
res = results[0]
print(f"第一个结果对象信息：{res}")

## 5. 可视化推理结果
- 从 `res.boxes` 提取边界框坐标、置信度与类别
- 分别调用 `.xyxy`, `.conf`, `.cls` 属性以避免解包错误
- 在图像上绘制矩形框及文字标签

### 第一部分：提取数据

In [ ]:
# 提取数据：边界框坐标 (Nx4)、置信度 (N)、类别索引 (N)
xyxy = res.boxes.xyxy.cpu().numpy()    # 形状 (N,4)
conf = res.boxes.conf.cpu().numpy()    # 形状 (N,)
cls  = res.boxes.cls.cpu().numpy()     # 形状 (N,)
names = model.names                   # 类别名称字典
print(f"检测到 {xyxy.shape[0]} 个目标")

### 第二部分：在图像上绘制检测框与标签

In [ ]:
# 复制原图用于绘制
img_show = img_rgb.copy()
for (x1, y1, x2, y2), c, cl in zip(xyxy, conf, cls):
    x1, y1, x2, y2 = map(int, (x1, y1, x2, y2))
    label = f"{names[int(cl)]}: {c:.2f}"
    # 绘制绿色矩形框
    cv2.rectangle(img_show, (x1, y1), (x2, y2), (0,255,0), 2)
    # 计算文字背景尺寸并绘制
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
    cv2.rectangle(img_show, (x1, y1-th-4), (x1+tw, y1), (0,255,0), -1)
    cv2.putText(img_show, label, (x1, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 1)

# 显示结果图
plt.figure(figsize=(6,6))
plt.axis('off')
plt.title('推理结果可视化')
plt.imshow(img_show)
plt.show()